In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/pranithaseemalamudi@my.unt.edu/consolidated_pipeline/1_setup/utilities

In [0]:
bronze_schema = "bronze"
silver_schema = "silver"
gold_schema = "gold"

In [0]:
print(bronze_schema, silver_schema, gold_schema)

In [0]:
catalog = "fmcg"
bronze_schema = "bronze"
silver_schema = "silver"
gold_schema = "gold"
data_source = "customers"

print(catalog, bronze_schema, silver_schema, gold_schema, data_source)


In [0]:
catalog = dbutils.widgets.get("catalog") or "default_catalog"
data_source = dbutils.widgets.get("data_source") or "default_source"

base_path = f's3://secondary-barsports/{data_source}/*.csv' 
print(base_path)

In [0]:
catalog = "fmcg"
data_source = "customers"
base_path = f's3://sportsbar-final/{data_source}/*.csv'
print(base_path)


In [0]:
df = (
    spark.read.format("csv")
        .option("header", True)
        .option("inferSchema", True)
        .load("s3://secondary-barsports/customers/")
        .withColumn("read_timestamp", F.current_timestamp())
        .select(
            "*",
            F.col("_metadata.file_name").alias("file_name"),
            F.col("_metadata.file_size").alias("file_size"),
        )
)

display(df.limit(10))

In [0]:
df.printSchema()

In [0]:
%run /Workspace/Users/pranithaseemalamudi@my.unt.edu/consolidated_pipeline/1_setup/utilities


In [0]:
(df.write
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")
)

In [0]:
# Create schemas if they don't exist
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.bronze")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.silver")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.gold")

In [0]:
# Check what catalogs are available
display(spark.sql("SHOW CATALOGS"))

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.bronze")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.silver")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.gold")

print(f"Schemas created under catalog: {catalog}")

In [0]:
# 1. Set variables
catalog = "fmcg"
data_source = "customers"
bronze_schema = "bronze"
silver_schema = "silver"
gold_schema = "gold"

# 2. Create schemas
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.bronze")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.silver")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.gold")

# 3. Verify
print(catalog, bronze_schema, silver_schema, gold_schema)

Sliver Processing 

In [0]:
df_bronze = spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.{data_source}")
df_bronze.show(10)

In [0]:
df_bronze.printSchema()


In [0]:
df_duplicates = (df_bronze
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
)

display(df_duplicates)

In [0]:
print('Rows after duplication dropped: ',df_bronze.count())
df_silver =df_bronze.dropDuplicates(["customer_id"])
print('Rows after duplication dropped: ',df_silver.count())

In [0]:
# remove whitespace
display(
    df_silver.filter(F.col("customer_name") != F.trim(F.col("customer_name")))
)

In [0]:
df_silver = df_silver.withColumn(
    "customer_name",
    F.trim(F.col("customer_name"))
)

In [0]:
df_silver.select('city').distinct().show()

In [0]:
# # typo dictionary
# city_typos = {
#     'Bengaluru': ['Bengaluruu', 'Bengaluruu', 'Bengalore'],
#     'Hyderabad': ['Hyderabadd', 'Hyderbad'],
#     'New Delhi': ['NewDelhi', 'NewDheli', 'NewDelhee']
# }

city_mapping = {
    'Bengaluruu': 'Bengaluru',
    'Bengalore': 'Bengaluru',

    'Hyderabadd': 'Hyderabad',
    'Hyderbad': 'Hyderabad',

    'NewDelhi': 'New Delhi',
    'NewDheli': 'New Delhi',
    'NewDelhee': 'New Delhi'
}


allowed = ["Bengaluru", "Hyderabad", "New Delhi"]

df_silver = (
    df_silver
    .replace(city_mapping, subset=["city"])
    .withColumn(
        "city",
        F.when(F.col("city").isNull(), None)
         .when(F.col("city").isin(allowed), F.col("city"))
         .otherwise(None)
    )
)




In [0]:
df_silver.select('city').distinct().show()


Capitalize Fix

In [0]:
df_silver.select('customer_name').distinct().show()

In [0]:
# Title case fix
df_silver = df_silver.withColumn(
    "customer_name",
    F.when(F.col("customer_name").isNull(), None)
     .otherwise(F.initcap("customer_name"))
)

In [0]:
# sanity check

df_silver.select('customer_name').distinct().show()

Missing Citites

In [0]:
df_silver.filter(F.col("city").isNull()).show(truncate=False)


In [0]:
null_customer_names = ['Sprintx Nutrition', 'Zenathlete Foods', 'Primefuel Nutrition', 'Recovery Lane']
df_silver.filter(F.col("customer_name").isin(null_customer_names)).show(truncate=False)

In [0]:

# Business Confirmation Note: City corrections confirmed by business team
customer_city_fix = {
    # Sprintx Nutrition
    789403: "New Delhi",

    # Zenathlete Foods
    789420: "Bengaluru",

    # Primefuel Nutrition
    789521: "Hyderabad",

    # Recovery Lane
    789603: "Hyderabad"
}

df_fix = spark.createDataFrame(
    [(k, v) for k, v in customer_city_fix.items()],
    ["customer_id", "fixed_city"]
)

display(df_fix)

In [0]:
df_silver = (
    df_silver
    .join(df_fix, "customer_id", "left")
    .withColumn(
        "city",
        F.coalesce("city", "fixed_city")   # Replace null with fixed city
    )
    .drop("fixed_city")
)

display(df_silver)

In [0]:
# Sanity Checks

null_customer_names = ['Sprintx Nutrition', 'Zenathlete Foods', 'Primefuel Nutrition', 'Recovery Lane']
df_silver.filter(F.col("customer_name").isin(null_customer_names)).show(truncate=False)

Converting Datatypes (Customer ID)

In [0]:
df_silver = df_silver.withColumn("customer_id", F.col("customer_id").cast("string"))
print(df_silver.printSchema())

### Standardizing Customer Attributes to Match Parent Company Data Model

In [0]:
df_silver = (
    df_silver
    # Build final customer column: "CustomerName-City" or "CustomerName-Unknown"
    .withColumn(
        "customer",
        F.concat_ws("-", "customer_name", F.coalesce(F.col("city"), F.lit("Unknown")))
    )
    
    # Static attributes aligned with parent data model
    .withColumn("market", F.lit("India"))
    .withColumn("platform", F.lit("Sports Bar"))
    .withColumn("channel", F.lit("Acquisition"))
)

In [0]:
display(df_silver.limit(5))

In [0]:
df_silver.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .option("mergeSchema", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

Gold Processing 

In [0]:
df_silver = spark.sql(f"SELECT * FROM {catalog}.{silver_schema}.{data_source};")
# take req cols only
df_gold = df_silver.select("customer_id", "customer_name", "city", "customer", "market", "platform", "channel")

In [0]:
df_gold.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")

**Mergeing Data From Parent House**

In [0]:
delta_table = DeltaTable.forName(spark, "fmcg.gold.dim_customers")
df_child_customers = spark.table("fmcg.gold.sb_dim_customers").select(
    F.col("customer_id").alias("customer_code"),
    "customer",
    "market",
    "platform",
    "channel"
)

In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "fmcg.gold.dim_customers")

df_child_customers = spark.table("fmcg.gold.sb_dim_customers").select(
    F.col("customer_id").alias("customer_code"),
    "customer",
    "market",
    "platform",
    "channel"
)

delta_table.alias("target").merge(
    source=df_child_customers.alias("source"),
    condition="target.customer_code = source.customer_code"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [0]:
result = delta_table.alias("target").merge(
    source=df_child_customers.alias("source"),
    condition="target.customer_code = source.customer_code"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

display(result)


In [0]:
display(spark.sql("SHOW TABLES IN fmcg.gold LIKE 'sb*'"))

In [0]:
display(spark.sql("SHOW TABLES IN fmcg.gold"))

In [0]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS fmcg.gold.dim_customers
    USING DELTA
    TBLPROPERTIES (delta.enableChangeDataFeed = true)
    AS
    SELECT 
        customer_id AS customer_code,
        customer,
        market,
        platform,
        channel
    FROM fmcg.gold.sb_dim_customers
""")

print("✅ Table created")

In [0]:
delta_table.alias("target").merge(
    source=df_child_customers.alias("source"),
    condition="target.customer_code = source.customer_code"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [0]:
delta_table.alias("target").merge(
    source=df_child_customers.alias("source"),
    condition="target.customer_code = source.customer_code"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()